### Processing of the main drug database to get information on blood pressure measurments:
At the bottom of this notebook there is code used to explore the BP data

In [ ]:
import pyspark
import dxpy
import dxdata
import json
import numpy as np
from bokeh.io import show, output_notebook
from bokeh.layouts import gridplot
import random
output_notebook()

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)

In [ ]:
%load_ext autoreload
%autoreload 2

from recode_anno_filter import (anno_bp,
                                filter_bp,
                                filter_meas,
                                show_stats,
                                anno_presc)

In [ ]:
db_name = "arb_db"
db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']

In [ ]:
import hail as hl
hl.init(sc=sc, default_reference='GRCh38')

read the main table:

In [ ]:
url = f"dnax://{db_uri}/base.ht"
full = hl.read_table(url)

### Annotate the full table with codes for BP drugs and measurments:

In [ ]:
full = anno_bp(full)

### Filter the table to only contain BP measurments  only BP medication info and save it:

This step includes filtering to the following codes only:
- O/E - blood pressure
- O/E-blood pressure reading NOS
- ABP - Arterial blood pressure
- Sitting systolic blood pressure
- Sitting diastolic blood pressure
- Standing systolic blood pressure
- Standing diastolic blood pressure
- Sitting blood pressure reading
- Standing blood pressure reading
- Systolic blood pressure
- DBP - Diastolic blood pressure
- Lying systolic blood pressure
- Lying diastolic blood pressure
- Lying blood pressure reading

In [ ]:
full = filter_bp(full)

In [ ]:
db_name = "arb_db"
tb_name = "bp_prel_test.ht"

db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']
url = f"dnax://{db_uri}/{tb_name}"

full.write(url, overwrite = True)
bp = hl.read_table(url) # this is an intermediate table, one can also start from this point after it is saved

### Parse recorded BP measurments, add bp measurments from assesment center and extract participants ages at events

In [ ]:
bp_meas = filter_meas(bp, spark)

### Save or load the intermediate table with measurments

In [ ]:
db_name = "arb_db"
tb_name = "bp_meas.ht"

db_uri = dxpy.find_one_data_object(name=f"{db_name}", classname="database")['id']
url = f"dnax://{db_uri}/{tb_name}"

bp_meas.write(url, overwrite = True)
bp_meas = hl.read_table(url)  #load this as intermediate file

### Optional section: show example stats from GP and ASCEN measurments:

In [ ]:
show_stats(bp_meas)

In [ ]:
show_stats(bp_meas) #old

In [ ]:
show_stats(bp_meas)

In [ ]:
show_stats(bp_meas) #old

### Optional section: evaluate differences between GP and ASCEN measurments [correlation]

### Add prescription info to each measurment and export

In [ ]:
bp_meas_with_zero = anno_presc(bp, bp_meas, db_uri, include_day_zero=True)

In [ ]:
bp_meas_with_zero.export('bp-with-zero.tsv')

In [ ]:
bp_meas_no_zero = anno_presc(bp, bp_meas, db_uri, include_day_zero=False)

In [ ]:
bp_meas_no_zero.export('bp-no-zero.tsv')